# Train Duck on the A30

Duck is a roughly two-million-parameter intent router for farm planning and remote satellite catalogs. The training examples are synthetic routing phrases. The model learns which tool to call. Weather summaries and planting comparisons use publisher data queried at request time.

## 1. Set up the kernel

Upload this notebook, `duck_router.py`, and `duck_service.py` into the same Jupyter folder. Select a Python kernel with CUDA-enabled PyTorch installed. You can run this notebook without uploading the project data folder.

In [ ]:
from pathlib import Path

import torch
from duck_router import MODEL_PATH, Router, predict, save, train

assert torch.cuda.is_available(), "Select the CUDA-enabled Python kernel on the A30."
device = torch.device("cuda")
print(torch.cuda.get_device_name(device))
print(f"Trainable parameters: {sum(p.numel() for p in Router().parameters()):,}")

## 2. Train and check routing

Review the outputs on these hand-written smoke prompts. A routing check does not measure forecast accuracy or agronomic suitability.

In [ ]:
model = train(device)
checks = {
    "Compare planting on 04-15 and 05-01 for maize": "planting",
    "Locate root zone moisture datasets in Kenya": "smap",
    "Find an ECOSTRESS pass in California last year": "ecostress",
    "What was the daily NASA POWER rainfall in 2024?": "power",
    "Write an essay about Mozart": "unknown",
    "Show NASA FIRMS fire detections near Fresno after the heat wave": "fire",
    "Did IMERG record heavy rain near Hilo last Thursday?": "gpm",
    "Can Landsat show how green the orchards were in May?": "landsat",
    "Look up the MOD11A1 daytime heat record for this county": "modis",
    "Find a NASA EMIT scene with surface reflectance for this field": "emit",
}
for question, expected in checks.items():
    intent, score = predict(model, question)
    print(f"{intent:12} {score:.3f}   expected {expected:12}   {question}")

## 3. Export and use Duck

Download `router.pt` from the cell output or the Jupyter file browser and put it beside `duck_service.py` on the dashboard machine. For local serving, install PyTorch and run `python duck_service.py` alongside `npm run dev` from OperationDucks.

For serving on the VM, run `python duck_service.py` on the VM or use the optional server cell below, then forward TCP port 8765 to the local dashboard machine over SSH using your VM's SSH connection details.

In [ ]:
from IPython.display import FileLink, display
from duck_router import load

save(model)
verified = load()
passed = 0
for question, expected in checks.items():
    actual, score = predict(verified, question)
    assert actual == predict(model, question)[0], f"Checkpoint mismatch: {question}"
    success = actual == expected and (expected == "unknown" or score >= 0.65)
    passed += success
    print(f"{'PASS' if success else 'FAIL'} | {actual} | {question}")
print(f"Routing regression: {passed}/{len(checks)} passed")
assert passed == len(checks), "Routing regression failed"
print(MODEL_PATH)
display(FileLink(str(MODEL_PATH.relative_to(Path.cwd()))))

## 4. Try Duck in the dashboard

- **Planting comparison**. Set Place to `Fresno, California; Modesto, California`. Ask `Compare planting days=14 min_c=10 min_mm=25`. Replace the example thresholds with those for your crop and seed variety. Duck checks the 1st and 15th of each month and ranks two starts per place using five historical years of daily minimum air temperature and total rainfall. Add up to four MM-DD dates to compare specific starts. It does not calculate field soil conditions, irrigation, yield, or future weather.
- **Satellite search**. Set Place to a named region or Worldwide and ask `Find SMAP soil moisture in 2025`. The service returns catalog scenes and their observation times.
- **Historical weather**. Pick a map center or input a latitude and longitude, then ask `NASA POWER rainfall from 2025-04-01 to 2025-04-30`.

The service makes on-demand publisher requests and keeps the responses in memory while answering. It does not read the existing local dataset archives.

### Optional VM server

Run this cell only when serving from the Jupyter kernel. Restart the kernel to stop the server and release the port.

In [ ]:
# Uncomment to serve the exported model from this Jupyter kernel.
# from threading import Thread
# from duck_router import load
# from duck_service import serve
# server = serve(load())
# Thread(target=server.serve_forever, daemon=True).start()